In [3]:
import os
import json
import time
import numpy as np
import pandas as pd
from scipy import sparse
import optuna
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

CARPETA = "../data/e3"

df = pd.read_json(f"{CARPETA}/datos_E3.jsonl", lines=True)
df["fila"] = np.arange(len(df))

tfidf = sparse.load_npz(f"{CARPETA}/tfidf.npz")
extra = np.load(f"{CARPETA}/features_extra.npy")

# Tarea de DETECCIÓN: la etiqueta binaria (falacia o no falacia)
# Se asume que 'etiqueta_deteccion' o 'falacia' es la columna de detección
df_train = df[df.split == "train"]
df_dev   = df[df.split == "dev"]
df_test  = df[df.split == "test"]

def preparar_deteccion(d, matriz):
    X = matriz[d["fila"].values]
    y = d["etiqueta_deteccion"].values  # 1 si es falacia, 0 si no
    return X, y

Optuna para detección

In [7]:
def optimizar_deteccion(nombre, matriz, n_trials=50):
    X_tr, y_tr = preparar_deteccion(df_train, matriz)
    X_dv, y_dv = preparar_deteccion(df_dev, matriz)
    pesos = compute_sample_weight("balanced", y_tr)
    
    # Máscara para evaluar SOLO sobre Political en dev
    mask = (df_dev["corpus"] == "political").values

    def objetivo(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 500),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        }
        m = XGBClassifier(**params, random_state=0, n_jobs=-1)
        m.fit(X_tr, y_tr, sample_weight=pesos)
        
        pred = m.predict(X_dv[mask])
        return f1_score(y_dv[mask], pred, average="macro", zero_division=0)

    t0 = time.time()
    estudio = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=42))
    estudio.optimize(objetivo, n_trials=n_trials, show_progress_bar=True)
    
    print(f"[{nombre}] F1 dev Political (Detección) = {estudio.best_value:.3f} ({(time.time()-t0)/60:.1f} min)")
    print(estudio.best_params)

    with open(f"mejores_params_xgboost_deteccion_{nombre}.json", "w") as f:
        json.dump(estudio.best_params, f, indent=2)
    return estudio.best_params

# Ejecución de Optuna para las dos representaciones en Detección:
params_tfidf_det = optimizar_deteccion("tfidf", tfidf, n_trials=50)
params_w2v_det   = optimizar_deteccion("w2v_medio", extra, n_trials=50)

  0%|          | 0/50 [00:00<?, ?it/s]

[tfidf] F1 dev Political (Detección) = 0.720 (2.3 min)
{'n_estimators': 324, 'max_depth': 9, 'learning_rate': 0.1402755739473422, 'subsample': 0.971045593953951, 'colsample_bytree': 0.35786852617144904, 'min_child_weight': 1}


  0%|          | 0/50 [00:00<?, ?it/s]

[w2v_medio] F1 dev Political (Detección) = 0.728 (5.6 min)
{'n_estimators': 240, 'max_depth': 10, 'learning_rate': 0.28061621583281476, 'subsample': 0.85108991186, 'colsample_bytree': 0.5542935451797338, 'min_child_weight': 2}


In [8]:
os.makedirs("predicciones", exist_ok=True)

def entrenar_deteccion_variante(nombre, matriz, train_df, params_de=None, semillas=(0, 1, 2)):
    with open(f"mejores_params_xgboost_deteccion_{params_de or nombre}.json") as f:
        mejores = json.load(f)

    X_tr, y_tr = preparar_deteccion(train_df, matriz)
    X_dv, y_dv = preparar_deteccion(df_dev, matriz)
    X_te, y_te = preparar_deteccion(df_test, matriz)

    m_dv = (df_dev["corpus"] == "political").values
    m_te = (df_test["corpus"] == "political").values

    pesos = compute_sample_weight("balanced", y_tr)

    f1s_dv, f1s_te = [], []

    for s in semillas:
        m = XGBClassifier(**mejores, random_state=s, n_jobs=-1)
        m.fit(X_tr, y_tr, sample_weight=pesos)

        pred_dv = m.predict(X_dv[m_dv])
        pred_te = m.predict(X_te[m_te])

        f1s_dv.append(f1_score(y_dv[m_dv], pred_dv, average="macro", zero_division=0))
        f1s_te.append(f1_score(y_te[m_te], pred_te, average="macro", zero_division=0))

        salida = pd.concat([
            pd.DataFrame({"id": df_dev["id"].values[m_dv], "split": "dev",
                          "real": y_dv[m_dv], "predicho": pred_dv}),
            pd.DataFrame({"id": df_test["id"].values[m_te], "split": "test",
                          "real": y_te[m_te], "predicho": pred_te}),
        ])
        salida.to_csv(f"predicciones/deteccion__xgboost__{nombre}__s{s}.csv", index=False)

    print(f"=== DETECCIÓN: {nombre} ===")
    print(f"  F1 Macro DEV  (Political): {np.mean(f1s_dv):.3f} ± {np.std(f1s_dv):.3f}")
    print(f"  F1 Macro TEST (Political): {np.mean(f1s_te):.3f} ± {np.std(f1s_te):.3f}\n")

Comparativa de modelos para Detección

In [9]:
# 1. Detección con TF-IDF
entrenar_deteccion_variante("tfidf", tfidf, df_train, params_de="tfidf")

# 2. Detección con Vector Medio + Signos (302 cols)
entrenar_deteccion_variante("w2v_medio", extra, df_train)

# 3. Detección entrenando SOLO con el subconjunto Political
entrenar_deteccion_variante("solo_political", tfidf,
                            df_train[df_train["corpus"] == "political"], params_de="tfidf")

=== DETECCIÓN: tfidf ===
  F1 Macro DEV  (Political): 0.706 ± 0.012
  F1 Macro TEST (Political): 0.535 ± 0.013

=== DETECCIÓN: w2v_medio ===
  F1 Macro DEV  (Political): 0.699 ± 0.021
  F1 Macro TEST (Political): 0.610 ± 0.019

=== DETECCIÓN: solo_political ===
  F1 Macro DEV  (Political): 0.650 ± 0.006
  F1 Macro TEST (Political): 0.557 ± 0.008



In [10]:
# Comprobación de que el CSV de detección contiene únicamente textos de political
c = pd.read_csv("predicciones/deteccion__xgboost__tfidf__s0.csv")

print("--- RECUENTO POR SPLIT ---")
print(c["split"].value_counts())

print("\n--- PREFIJOS DE ID (Solo debe salir fpo = political) ---")
print(c["id"].str[:3].value_counts())

--- RECUENTO POR SPLIT ---
split
test    413
dev     401
Name: count, dtype: int64

--- PREFIJOS DE ID (Solo debe salir fpo = political) ---
id
fpo    635
fpn    179
Name: count, dtype: int64
